In [5]:
import pandas as pd

file_path = "C:/Users/Alban/Documents/IMDS/Projet_5A/Donnees/Matches_with_Elo.csv"

# Lecture correcte avec le bon séparateur
df = pd.read_csv(file_path, sep=";", parse_dates=["MatchDate"], low_memory=False)
print(f"Shape: {df.shape}")
print(df.head(3))

# Vérification des colonnes clés
expected_core = [
    "MatchDate", "HomeTeam", "AwayTeam",
    "HomeEloSnap", "AwayEloSnap", "FTResult"
]
missing = [c for c in expected_core if c not in df.columns]
if missing:
    raise KeyError(f"Colonnes essentielles manquantes : {missing}")

print("\nColonnes disponibles :")
print(sorted(df.columns.tolist())[:25])
print("...")

print("\nTaux de valeurs manquantes (top 15) :")
print(df.isna().mean().sort_values(ascending=False).head(15))

print("\nDistribution de FTResult :")
print(df["FTResult"].value_counts(normalize=True).round(3))


C:\Users\Alban\AppData\Local\Temp\ipykernel_25324\4173533404.py:6: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df = pd.read_csv(file_path, sep=";", parse_dates=["MatchDate"], low_memory=False)


Shape: (165619, 66)
  Division  MatchDate MatchTime        HomeTeam      AwayTeam  Form3Home  \
0       T1 2003-08-17       NaN     Malatyaspor  A. Sebatspor        9.0   
1       T1 2003-08-31       NaN     Trabzonspor  A. Sebatspor        4.0   
2       T1 2003-09-21       NaN  Diyarbakirspor  A. Sebatspor        0.0   

   Form5Home  Form3Away  Form5Away  FTHome  ...  PDraw_norm PAway_norm  \
0       12.0        0.0        0.0     3.0  ...    0.236453   0.147783   
1        7.0        0.0        0.0     2.0  ...    0.205161   0.126907   
2        3.0        3.0        3.0     1.0  ...    0.261319   0.222121   

   ShotsDifference  CornersDifference CardsHome  CardsAway  CardsDiff  \
0              NaN                NaN       NaN        NaN        NaN   
1              NaN                NaN       NaN        NaN        NaN   
2              NaN                NaN       NaN        NaN        NaN   

   GameDominanceIndex  Country  country.1  
0                 NaN      TUR        TUR

In [6]:
import numpy as np

# ✅ Liste des variables explicatives (pré-match)
features_base = [
    # --- ELO ---
    "HomeEloSnap", "AwayEloSnap", "EloDiff", "EloTotal", "EloAdvantage",
    "EloChange1M_Diff", "EloChange3M_Diff", "EloChange6M_Diff",

    # --- Forme ---
    "Form3Home", "Form5Home", "Form3Away", "Form5Away",
    "Form3Diff", "Form5Diff", "FormMomentumHome", "FormMomentumAway",

    # --- Cotes / Probabilités implicites ---
    "OddHome", "OddDraw", "OddAway",
    "PHome_norm", "PDraw_norm", "PAway_norm", "ProbSum",

    # --- Indicateurs contextuels ---
    "GameDominanceIndex", "HomeAdvantage"
]

# Certaines colonnes peuvent manquer si non présentes dans ton CSV (par ex HomeAdvantage)
features = [c for c in features_base if c in df.columns]
print(f"{len(features)} features retenues : {features}")

# Vérification des NaN
nan_ratio = df[features].isna().mean().sort_values(ascending=False)
print("\nTaux de NaN :")
print(nan_ratio.head(10))

# Visualisation statistique basique
print("\nRésumé statistique (moyenne ± écart-type) des principales features :")
display(df[features].describe().T[["mean", "std"]].head(10))


24 features retenues : ['HomeEloSnap', 'AwayEloSnap', 'EloDiff', 'EloTotal', 'EloAdvantage', 'EloChange1M_Diff', 'EloChange3M_Diff', 'EloChange6M_Diff', 'Form3Home', 'Form5Home', 'Form3Away', 'Form5Away', 'Form3Diff', 'Form5Diff', 'FormMomentumHome', 'FormMomentumAway', 'OddHome', 'OddDraw', 'OddAway', 'PHome_norm', 'PDraw_norm', 'PAway_norm', 'ProbSum', 'GameDominanceIndex']

Taux de NaN :
GameDominanceIndex    0.430899
EloChange6M_Diff      0.228537
EloChange3M_Diff      0.201348
EloChange1M_Diff      0.185703
EloDiff               0.093667
EloAdvantage          0.093667
EloTotal              0.093667
AwayEloSnap           0.084199
HomeEloSnap           0.027171
PHome_norm            0.016085
dtype: float64

Résumé statistique (moyenne ± écart-type) des principales features :


c:\Users\Alban\Documents\IMDS\Projet_5A\.venv_pred_match\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,mean,std
HomeEloSnap,1509.773280,158.119502
AwayEloSnap,1518.213367,156.721833
EloDiff,0.036056,130.279247
EloTotal,3039.337291,284.982320
EloAdvantage,0.000015,0.041826
EloChange1M_Diff,0.026752,22.007096
EloChange3M_Diff,0.084653,37.590934
EloChange6M_Diff,0.042312,52.943719
Form3Home,3.992496,2.376972
Form5Home,6.747400,3.265390


In [9]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import log_loss, accuracy_score, f1_score
import numpy as np
import pandas as pd
import warnings

# ✅ Retirer GameDominanceIndex pour le moment (trop de NaN)
features_model = [f for f in features if f != "GameDominanceIndex"]

# ✅ Préprocesseur : imputation + standardisation
preproc = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), features_model)
])

# ✅ Modèle
model_lr = LogisticRegression(
    multi_class="multinomial",
    solver="lbfgs",
    C=1.0,
    max_iter=300,
    random_state=42
)

# ✅ Split temporel
def make_time_splits(dates, n_splits=5, min_train_size=20000):
    order = np.argsort(dates.values)
    n = len(order)
    fold_sizes = np.full(n_splits, n // n_splits, dtype=int)
    fold_sizes[: n % n_splits] += 1
    bounds = np.cumsum(fold_sizes)

    splits = []
    start = 0
    for i in range(n_splits - 1):
        val_start = bounds[i] - fold_sizes[i]
        val_end = bounds[i]
        train_idx_sorted = np.arange(start, val_start)
        val_idx_sorted = np.arange(val_start, val_end)
        if len(train_idx_sorted) < min_train_size:
            continue
        train_idx = order[train_idx_sorted]
        val_idx = order[val_idx_sorted]
        splits.append((train_idx, val_idx))
    return splits

splits = make_time_splits(df["MatchDate"], n_splits=5, min_train_size=20000)
print(f"{len(splits)} folds temporels créés ✅")

# ✅ Fonction Brier multiclasses
def multiclass_brier_score(y_true, proba, labels):
    lab2idx = {lab: i for i, lab in enumerate(labels)}
    y_idx = np.array([lab2idx[y] for y in y_true])
    y_onehot = np.zeros_like(proba)
    y_onehot[np.arange(len(y_true)), y_idx] = 1.0
    return np.mean((proba - y_onehot) ** 2)

# ✅ Boucle d’évaluation
labels = ["H", "D", "A"]
records = []

# --- Remplacement des inf/-inf par NaN ---
df[features_model] = df[features_model].replace([np.inf, -np.inf], np.nan)

# --- Vérification (facultative) ---
print("Valeurs infinies détectées après nettoyage :", np.isinf(df[features_model].to_numpy()).sum())

# --- Capping optionnel pour valeurs aberrantes (5e - 95e percentile)
for col in features_model:
    if df[col].dtype.kind in "iuf":  # seulement numériques
        low, high = df[col].quantile(0.01), df[col].quantile(0.99)
        df[col] = df[col].clip(lower=low, upper=high)

records = []

for fold_id, (tr_idx, va_idx) in enumerate(splits, start=1):
    X_train = df.loc[tr_idx, features_model]
    y_train = df.loc[tr_idx, "FTResult"]

    # Filtrer la validation pour ne garder que les matchs terminés
    val_fold = df.loc[va_idx, ["FTResult"] + features_model].copy()
    val_fold = val_fold[val_fold["FTResult"].isin(["H", "D", "A"])]
    X_val = val_fold[features_model]
    y_val = val_fold["FTResult"]

    if len(y_val) == 0:
        print(f"[Fold {fold_id}] Aucun match terminé dans cette période → ignoré.")
        continue

    pipe = Pipeline([
        ("preproc", preproc),
        ("clf", model_lr)
    ])
    pipe.fit(X_train, y_train)

    # Calibration isotonic (si possible)
    try:
        cal = CalibratedClassifierCV(pipe.named_steps["clf"], method="isotonic", cv="prefit")
        cal.fit(pipe.named_steps["preproc"].transform(X_train), y_train)
        proba_val = cal.predict_proba(pipe.named_steps["preproc"].transform(X_val))
    except Exception as e:
        warnings.warn(f"Calibration impossible ({e}) → proba brutes utilisées.")
        proba_val = pipe.predict_proba(X_val)

    y_pred = np.array([labels[np.argmax(p)] for p in proba_val])

    ll = log_loss(y_val, proba_val, labels=labels)
    acc = accuracy_score(y_val, y_pred)
    f1m = f1_score(y_val, y_pred, average="macro")
    brier = multiclass_brier_score(y_val, proba_val, labels)

    records.append({
        "fold": fold_id,
        "logloss": ll,
        "accuracy": acc,
        "f1_macro": f1m,
        "brier": brier
    })
    print(f"[Fold {fold_id}] LogLoss={ll:.4f} | Acc={acc:.3f} | F1={f1m:.3f} | Brier={brier:.4f}")

metrics_lr = pd.DataFrame(records)
print("\n=== Moyennes ± écarts-types ===")
print(metrics_lr.agg(["mean", "std"]).round(4))

3 folds temporels créés ✅
Valeurs infinies détectées après nettoyage : 0


c:\Users\Alban\Documents\IMDS\Projet_5A\.venv_pred_match\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Alban\Documents\IMDS\Projet_5A\.venv_pred_match\Lib\site-packages\sklearn\calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(
c:\Users\Alban\Documents\IMDS\Projet_5A\.venv_pred_match\Lib\site-packages\sklearn\metrics\_classification.py:210: UserWarning: Labels passed were ['H', 'D', 'A']. But this function assumes labels are ordered lexicographically. Pass the ordered labels=['A', 'D', 'H'] and ensure that the columns of y_prob correspond to this ordering.
  warnings.warn(


[Fold 1] LogLoss=1.0156 | Acc=0.227 | F1=0.175 | Brier=0.2779


c:\Users\Alban\Documents\IMDS\Projet_5A\.venv_pred_match\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Alban\Documents\IMDS\Projet_5A\.venv_pred_match\Lib\site-packages\sklearn\calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(
c:\Users\Alban\Documents\IMDS\Projet_5A\.venv_pred_match\Lib\site-packages\sklearn\metrics\_classification.py:210: UserWarning: Labels passed were ['H', 'D', 'A']. But this function assumes labels are ordered lexicographically. Pass the ordered labels=['A', 'D', 'H'] and ensure that the columns of y_prob correspond to this ordering.
  warnings.warn(


[Fold 2] LogLoss=1.0081 | Acc=0.235 | F1=0.175 | Brier=0.2775


ValueError: Input contains NaN